# Causal intervention diagnostic

This diagnostic separates retrieval failure from intervention-transfer failure. It supplies Mnemo with the known edit that owns each two-hop causal question, then uses the normal block-18 interceptor. No oracle behavior is used by benchmark generation.

The comparison has three conditions: base output, normal Mnemo routing, and forced oracle routing. The oracle only bypasses the router; it does not bypass edit-vector training, the interceptor, generation, or cleanup. Results are written under `data/causality/intervention/`, separate from evaluation artifacts.

In [ ]:
import hashlib
import json
import random
import re
import sys
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path

import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.editors.mnemo import Mnemo
from src.editors.editor_utils import EditRequest


In [ ]:
# Experiment controls. Increase limits only after the smoke run looks sensible.
MODEL_KEY = 'qwen2.5-7b'
MNEMO_CONFIG_PATH = PROJECT_ROOT / 'src/config/mnemo/qwen2.5-7b.yaml'
DATA_PATH = PROJECT_ROOT / 'data/datasets/uniedit/clean/causality.jsonl'
OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'oracle.json'
SEED = 0
QUERY_LIMIT_PER_STRATUM = 8
MAX_EDIT_BANK = 100
ORACLE_STRENGTHS = (0.25, 0.5, 0.75, 1.0)
BATCH_SIZE = 4
MAX_NEW_TOKENS = 24
REPETITION_PENALTY = 1.5

# The diagnostic deliberately starts in external-only mode.
EXTERNAL_ONLY = True

print(f'data={DATA_PATH}')
print(f'output={OUTPUT_PATH}')
print(f'query_limit_per_stratum={QUERY_LIMIT_PER_STRATUM}, bank_cap={MAX_EDIT_BANK}')


In [ ]:
def load_records(path: Path) -> list[dict]:
    records = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row.get('edit'), dict) or not isinstance(row.get('causality'), list):
                raise ValueError(f'Invalid causality row at line {line_number}')
            records.append(row)
    return records

def owner_key(edit: dict) -> tuple[str, str]:
    return (str(edit['subject']['id']), str(edit['relation']['id']))

def dependency_keys(query: dict) -> list[tuple[str, str]]:
    return [(str(item['subject_id']), str(item['relation_id'])) for item in query.get('dependency_edit_keys', [])]

records = load_records(DATA_PATH)
owners = {owner_key(row['edit']): index for index, row in enumerate(records)}
queries = []
for row_index, row in enumerate(records):
    for query_index, query in enumerate(row['causality']):
        keys = dependency_keys(query)
        # The default data policy is independent edits: one causal query has one owner.
        if len(keys) != 1 or keys[0] not in owners:
            continue
        queries.append({
            'row_index': row_index,
            'query_index': query_index,
            'query': query,
            'owner_key': keys[0],
            'owner_row_index': owners[keys[0]],
        })

print(f'loaded edits={len(records)}, eligible queries={len(queries)}')
print(Counter((item['query']['source_edit_position'], item['query']['origin']) for item in queries))


In [ ]:
def stratified_sample(items: list[dict], per_stratum: int, seed: int) -> list[dict]:
    groups = defaultdict(list)
    for item in items:
        query = item['query']
        groups[(query['source_edit_position'], query['origin'])].append(item)
    rng = random.Random(seed)
    selected = []
    for group_key in sorted(groups):
        group = list(groups[group_key])
        rng.shuffle(group)
        selected.extend(group[:per_stratum])
    selected.sort(key=lambda item: (item['row_index'], item['query_index']))
    return selected

selected_queries = stratified_sample(queries, QUERY_LIMIT_PER_STRATUM, SEED)
required_owner_keys = sorted({item['owner_key'] for item in selected_queries})
if len(required_owner_keys) > MAX_EDIT_BANK:
    raise ValueError(
        f'Selected queries require {len(required_owner_keys)} owners, exceeding MAX_EDIT_BANK={MAX_EDIT_BANK}. '
        'Lower QUERY_LIMIT_PER_STRATUM or increase the explicit bank cap.'
    )

distractor_keys = sorted(set(owners) - set(required_owner_keys))
random.Random(SEED + 1).shuffle(distractor_keys)
bank_keys = required_owner_keys + distractor_keys[:MAX_EDIT_BANK - len(required_owner_keys)]
random.Random(SEED + 2).shuffle(bank_keys)
bank_rows = [records[owners[key]] for key in bank_keys]
bank_index = {key: index for index, key in enumerate(bank_keys)}
for item in selected_queries:
    item['owner_memory_index'] = bank_index[item['owner_key']]

print(f'selected queries={len(selected_queries)}, edit bank={len(bank_rows)}')
print(Counter((item['query']['source_edit_position'], item['query']['origin']) for item in selected_queries))


In [ ]:
def build_mnemo() -> Mnemo:
    config = load_config(MNEMO_CONFIG_PATH)
    config.external_only = EXTERNAL_ONLY
    return Mnemo(config)

def clean_text(text: str) -> str:
    text = text.strip().casefold()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

def generate_texts(
    editor: Mnemo,
    prompts: list[str],
    *,
    forced_indices: list[int] | None = None,
    strength: float | None = None,
) -> list[str]:
    if not prompts:
        return []
    if forced_indices is not None and len(forced_indices) != len(prompts):
        raise ValueError('Forced indices must align with prompts')
    tokenizer = editor.tokenizer
    model = editor.model
    original_padding = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    predictions = []
    try:
        for start in range(0, len(prompts), BATCH_SIZE):
            batch = prompts[start:start + BATCH_SIZE]
            batch_indices = None if forced_indices is None else forced_indices[start:start + BATCH_SIZE]
            inputs = tokenizer(batch, return_tensors='pt', padding=True).to(editor.device)
            generation_inputs = {
                **inputs,
                'max_new_tokens': MAX_NEW_TOKENS,
                'repetition_penalty': REPETITION_PENALTY,
                'do_sample': False,
                'pad_token_id': tokenizer.pad_token_id,
                'eos_token_id': tokenizer.eos_token_id,
                'routing_prompts': batch,
            }
            patch_context = (
                nullcontext()
                if batch_indices is None
                else editor.force_patch(batch_indices, strength=strength)
            )
            with patch_context, torch.inference_mode():
                output_ids = model.generate(**generation_inputs)
            prompt_length = inputs['input_ids'].shape[1]
            predictions.extend(tokenizer.batch_decode(output_ids[:, prompt_length:], skip_special_tokens=True))
    finally:
        tokenizer.padding_side = original_padding
    return [prediction.strip() for prediction in predictions]

def answer_match(prediction: str, answers: list[str]) -> tuple[bool, str | None]:
    value = clean_text(prediction)
    if not value:
        return False, None
    normalized = [clean_text(answer) for answer in answers if clean_text(answer)]
    for answer in normalized:
        if value == answer or value.startswith(answer + ' '):
            return True, 'prefix'
    for answer in normalized:
        if f' {answer} ' in f' {value} ':
            return True, 'containment'
    return False, None


In [ ]:
def apply_owner_edits(editor: Mnemo) -> None:
    for row in tqdm(bank_rows, desc='Applying owner edits'):
        edit = row['edit']
        editor.edit(EditRequest(
            subject=edit['subject']['str'],
            prompt=edit['prompt'],
            answer=edit['answers'][0],
        ))
    if editor.external_memory_size != len(bank_rows):
        raise RuntimeError('The Mnemo memory bank is not aligned with sampled owners')

def run_predictions(editor: Mnemo) -> tuple[list[str], list[str], dict[str, list[str]]]:
    query_prompts = [item['query']['prompt'] for item in selected_queries]
    base_predictions = generate_texts(editor, query_prompts)
    apply_owner_edits(editor)
    normal_predictions = generate_texts(editor, query_prompts)
    owner_indices = [item['owner_memory_index'] for item in selected_queries]
    oracle_predictions = {
        str(strength): generate_texts(
            editor,
            query_prompts,
            forced_indices=owner_indices,
            strength=strength,
        )
        for strength in ORACLE_STRENGTHS
    }
    return base_predictions, normal_predictions, oracle_predictions


In [ ]:
# Kept as a separate definition cell so the expensive run is explicit below.
def compare_scores(base_scores: list[dict], candidate_scores: list[dict]) -> dict:
    groups = defaultdict(lambda: {'count': 0, 'incorrect_to_correct': 0, 'correct_to_incorrect': 0, 'net_gain': 0})
    for item, base, candidate in zip(selected_queries, base_scores, candidate_scores, strict=True):
        query = item['query']
        keys = ('overall', f'position_{query["source_edit_position"]}')
        for key in keys:
            groups[key]['count'] += 1
            if not base['matched'] and candidate['matched']:
                groups[key]['incorrect_to_correct'] += 1
            if base['matched'] and not candidate['matched']:
                groups[key]['correct_to_incorrect'] += 1
    for values in groups.values():
        values['net_gain'] = values['incorrect_to_correct'] - values['correct_to_incorrect']
    return dict(sorted(groups.items()))


In [ ]:
def score_selected_predictions(
    base_predictions: list[str],
    normal_predictions: list[str],
    oracle_predictions: dict[str, list[str]],
) -> tuple[list[dict], list[dict], dict[str, list[dict]]]:
    base_scores = score_condition(base_predictions)
    normal_scores = score_condition(normal_predictions)
    oracle_scores = {strength: score_condition(predictions) for strength, predictions in oracle_predictions.items()}
    return base_scores, normal_scores, oracle_scores


In [ ]:
def score_condition(predictions: list[str]) -> list[dict]:
    scored = []
    for item, prediction in zip(selected_queries, predictions, strict=True):
        matched, mode = answer_match(prediction, item['query']['answers'])
        scored.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
    return scored

def summarize(scores: list[dict]) -> dict:
    groups = defaultdict(list)
    for item, score in zip(selected_queries, scores, strict=True):
        query = item['query']
        groups['overall'].append(score['matched'])
        groups[f'position_{query["source_edit_position"]}'].append(score['matched'])
        groups[f'{query["origin"]}:position_{query["source_edit_position"]}'].append(score['matched'])
    return {key: {'count': len(values), 'accuracy': sum(values) / len(values) if values else None} for key, values in sorted(groups.items())}


In [ ]:
def fingerprint(items: list[dict]) -> str:
    payload = '\n'.join(item['query']['prompt'] for item in items).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def atomic_save(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=True, indent=2), encoding='utf-8')
    temporary.replace(path)

def build_payload(base_predictions, normal_predictions, oracle_predictions, base_scores, normal_scores, oracle_scores) -> dict:
    result_rows = []
    for index, item in enumerate(selected_queries):
        query = item['query']
        result_rows.append({
            'query_index': index,
            'prompt': query['prompt'],
            'answers': query['answers'],
            'source_edit_position': query['source_edit_position'],
            'origin': query['origin'],
            'owner_key': list(item['owner_key']),
            'owner_memory_index': item['owner_memory_index'],
            'base_prediction': base_predictions[index],
            'normal_prediction': normal_predictions[index],
            'base_match': base_scores[index],
            'normal_match': normal_scores[index],
            'oracle': {
                strength: {**oracle_scores[strength][index], 'prediction': oracle_predictions[strength][index]}
                for strength in oracle_predictions
            },
        })
    return {
        'schema_version': 1,
        'experiment': 'causal_intervention_oracle',
        'model_key': MODEL_KEY,
        'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
        'seed': SEED,
        'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
        'edit_bank_size': len(bank_rows),
        'oracle_strengths': list(ORACLE_STRENGTHS),
        'dataset_fingerprint': fingerprint(selected_queries),
        'summary': {
            'base': summarize(base_scores),
            'normal': summarize(normal_scores),
            'oracle': {strength: summarize(scores) for strength, scores in oracle_scores.items()},
        },
        'comparisons': {
            'normal_vs_base': compare_scores(base_scores, normal_scores),
            'oracle_vs_base': {strength: compare_scores(base_scores, scores) for strength, scores in oracle_scores.items()},
        },
        'edit_bank': [
            {
                'memory_index': index,
                'owner_key': list(key),
                'subject': row['edit']['subject']['str'],
                'prompt': row['edit']['prompt'],
                'answer': row['edit']['answers'][0],
            }
            for index, (key, row) in enumerate(zip(bank_keys, bank_rows, strict=True))
        ],
        'rows': result_rows,
    }

def run_oracle_experiment() -> dict:
    editor = build_mnemo()
    try:
        base_predictions, normal_predictions, oracle_predictions = run_predictions(editor)
        base_scores, normal_scores, oracle_scores = score_selected_predictions(
            base_predictions, normal_predictions, oracle_predictions
        )
        payload = build_payload(
            base_predictions, normal_predictions, oracle_predictions,
            base_scores, normal_scores, oracle_scores,
        )
        atomic_save(payload, OUTPUT_PATH)
        print('base', payload['summary']['base'])
        print('normal', payload['summary']['normal'])
        for strength, summary in payload['summary']['oracle'].items():
            print(f'oracle strength={strength}', summary)
        print(f'saved {OUTPUT_PATH}')
        return payload
    finally:
        editor.reset()


Interpretation: if oracle accuracy is high but normal routing is low, the main problem is retrieval. If both are low, the retrieved edit vector does not transfer reliably to the causal prompt. Position 0 and position 1 are reported separately because a second-hop edit is not necessarily named in the causal question.

In [ ]:
# Uncomment the action you want to run.
# result = run_oracle_experiment()
